# InsightAid AI — Sentiment Model Training (Phase 6, FYP2.md §46-49)

**Status: scaffolding.** No labelled training data ships with this repo. Every
cell below is real, runnable code, but until you point it at an actual CSV
export (see `ml/sentiment/data/schema.md`), it produces nothing — there is
no "demo run" with placeholder data, because that would risk this notebook
being mistaken for a real result. Do not hand-write numbers into any cell's
output and keep them; every metric this notebook produces should come from
an actual `evaluate.py` run against real predictions.

**Primary model:** XLM-RoBERTa (FYP2.md §46). **Comparison models:** mBERT
and a TF-IDF+LogisticRegression baseline — these two, and the
translate-to-English-first vs. classify-in-original-language comparison
below, are this project's own methodology addition, not part of FYP2.md's
literal spec (confirmed absent from it). See `../README.md` and
`docs/SENTIMENT_MODEL.md` for the full write-up.

**Training happens here (Colab/GPU), never inside a Vercel request or a
Supabase Edge Function**, per §46's explicit instruction.


## 1. Setup

In [ ]:
# Colab: uncomment to install. Locally, `pip install -r ../requirements.txt` instead.
# !pip install -q -r ../requirements.txt


In [ ]:
import sys
from pathlib import Path

# Makes `ml/sentiment` importable as the `src` package, whether this notebook
# runs from Colab (after cloning/uploading the repo) or locally.
REPO_ROOT_CANDIDATES = [Path.cwd().parent, Path.cwd()]
for candidate in REPO_ROOT_CANDIDATES:
    if (candidate / "src" / "weak_labels.py").exists():
        sys.path.insert(0, str(candidate))
        break

import pandas as pd

from src import metrics as metrics_module
from src import translate as translate_module
from src.clean import preprocess_text
from src.evaluate import run_full_comparison
from src.split import stratified_split
from src.train_tfidf_baseline import predict as tfidf_predict
from src.train_tfidf_baseline import train_tfidf_baseline
from src.weak_labels import classify_by_rating

# Heavy deps — only needed once you actually train a transformer.
from src.train_mbert import train_mbert
from src.train_xlmr import predict as transformer_predict
from src.train_xlmr import train_transformer, XLMR_CHECKPOINT
from src.train_mbert import MBERT_CHECKPOINT


## 2. Load data

Set `DATA_CSV` to a real export matching `data/schema.md` before running
anything below. `MANUAL_VAL_CSV` is optional — a human-annotated subset,
same schema but with gold `label` values.


In [ ]:
DATA_CSV = "REPLACE_WITH_REAL_EXPORT_PATH.csv"  # see ../data/schema.md
MANUAL_VAL_CSV = None  # optional: path to a manually-labelled subset, same schema

assert Path(DATA_CSV).exists(), (
    "No training data at DATA_CSV. This notebook is scaffolding until you "
    "point it at a real export of processed_data (see ../data/schema.md). "
    "Do not fabricate rows to make this cell pass."
)

df = pd.read_csv(DATA_CSV)
manual_df = pd.read_csv(MANUAL_VAL_CSV) if MANUAL_VAL_CSV else None
print(f"Loaded {len(df)} rows.")


## 3. Weak labels + preprocessing parity

`classify_by_rating` mirrors the JS app's `rating_heuristic_v1` exactly (see
`src/weak_labels.py`'s docstring). Rows that already have a manual `label`
keep it; rows with only a `rating` get a weak label. Rows with neither are
dropped — there's nothing to train on.


In [ ]:
def resolve_label(row):
    if pd.notna(row.get("label")):
        return row["label"]
    weak = classify_by_rating(row.get("rating"))
    return weak.sentiment if weak else None

df["label"] = df.apply(resolve_label, axis=1)
df = df.dropna(subset=["label", "cleaned_text", "language"]).reset_index(drop=True)
print(df["label"].value_counts())
print(df["language"].value_counts())


## 4. Leakage-safe stratified split

In [ ]:
split_result = stratified_split(df, manual_df=manual_df)
print(split_result.leakage_report)
print(f"dropped {split_result.dropped_duplicates} exact-duplicate rows")

train_df, val_df, test_df = split_result.train, split_result.val, split_result.test
assert split_result.leakage_report["clean"], "Leakage check failed — do not proceed to training."


## 5. Train all three models

XLM-R (primary), mBERT (comparison), TF-IDF+LogisticRegression (baseline).
Adjust `epochs`/`batch_size` for your available compute.


In [ ]:
MODEL_VERSION = "v0-CHANGE-ME"  # bump this per real training run
RESULTS_DIR = Path("../results")

xlmr_trained = train_transformer(
    XLMR_CHECKPOINT, train_df, val_df, RESULTS_DIR / "xlmr_model", num_epochs=3, batch_size=16
)


In [ ]:
mbert_trained = train_mbert(train_df, val_df, RESULTS_DIR / "mbert_model", num_epochs=3, batch_size=16)


In [ ]:
tfidf_model = train_tfidf_baseline(train_df["cleaned_text"], train_df["label"])


## 6. Evaluate — both paths, all three models

`run_full_comparison` scores every (model, eval_path) pair on the held-out
test set and writes a real metrics JSON per pair to `../results/`.


In [ ]:
predict_fns = {
    "xlmr": lambda texts: transformer_predict(xlmr_trained, texts),
    "mbert": lambda texts: transformer_predict(mbert_trained, texts),
    "tfidf_lr": lambda texts: tfidf_predict(tfidf_model, texts),
}

rows = run_full_comparison(
    predict_fns,
    test_df,
    model_version=MODEL_VERSION,
    dataset_version=None,  # set to a real dataset export version/hash if you track one
    training_samples=len(train_df),
    validation_samples=len(val_df),
    results_dir=RESULTS_DIR,
)

for row in rows:
    print(f"{row.model_type:10s} {row.eval_path:22s} acc={row.accuracy} f1={row.f1}")


## 7. Record results in `model_versions`

The `model_versions` table only accepts writes from the Supabase **service
role** (see `supabase/migrations/0004_model_versions.sql`'s RLS policy) —
never from the frontend, and not from an anon/authenticated key here either.
Paste your own service role key into a Colab secret (never into this
notebook's source) if you want to upload automatically; otherwise the printed
rows below are ready to insert by hand via the SQL Editor.


In [ ]:
from dataclasses import asdict
import json as _json

for row in rows:
    print(_json.dumps(asdict(row), indent=2))

# Example (uncomment and fill in your own values — do not commit a real key):
# from supabase import create_client
# supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)  # from a Colab secret
# for row in rows:
#     supabase.table("model_versions").insert(asdict(row)).execute()
